# 03 — Preprocessing
Clean, encode, scale and split the data.

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib, os

df = pd.read_csv('../data/processed/listings_eda.csv')
print(df.shape)
print(df.columns.tolist())
print(df.shape)

(2592, 12)
['price', 'accommodates', 'bedrooms', 'bathrooms', 'neighbourhood_cleansed', 'room_type', 'minimum_nights', 'latitude', 'longitude', 'availability_365', 'number_of_reviews', 'reviews_per_month']
(2592, 12)


In [18]:
# Keep only useful features
FEATURES = ['accommodates', 'bedrooms', 'bathrooms',
            'neighbourhood_cleansed', 'room_type',
            'minimum_nights', 'latitude', 'longitude',
            'availability_365', 'number_of_reviews',
            'reviews_per_month']
TARGET = 'price'

df = df[FEATURES + [TARGET]].copy()
df = df.dropna(subset=[TARGET])
df = df[df[TARGET] > 0]
print(f'After price filter: {df.shape}')

After price filter: (2592, 12)


In [19]:
# Fill missing numerics
for col in ['bedrooms', 'bathrooms', 'reviews_per_month']:
    df[col] = df[col].fillna(df[col].median())

# One-hot encode categoricals
df = pd.get_dummies(df, columns=['neighbourhood_cleansed', 'room_type'], drop_first=True)
print(f'After encoding: {df.shape}')
df.head()

After encoding: (2592, 46)


,accommodates,bedrooms,bathrooms,minimum_nights,latitude,longitude,availability_365,number_of_reviews,reviews_per_month,price,...,neighbourhood_cleansed_Sihlfeld,neighbourhood_cleansed_Unterstrass,neighbourhood_cleansed_Weinegg,neighbourhood_cleansed_Werd,neighbourhood_cleansed_Wipkingen,neighbourhood_cleansed_Witikon,neighbourhood_cleansed_Wollishofen,room_type_Hotel room,room_type_Private room,room_type_Shared room
0,2,1.0,1.5,3,47.35761,8.52131,269,0,0.885,232.0,...,False,False,False,False,False,False,False,False,False,False
1,1,1.0,1.0,5,47.36514,8.52615,92,9,0.050,48.0,...,False,False,False,False,False,False,False,False,True,False
2,3,2.0,1.5,4,47.38942,8.51881,280,31,0.180,501.0,...,False,False,False,False,False,False,False,False,False,False
3,4,2.0,1.0,4,47.37372,8.54452,13,383,2.300,160.0,...,False,False,False,False,False,False,False,False,False,False
4,6,3.0,2.0,7,47.36713,8.54868,28,13,0.080,360.0,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
X = df.drop(columns=[TARGET])
y = np.log1p(df[TARGET])  # log-transform: stabilises right-skewed price distribution

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [21]:
scaler = StandardScaler()
num_cols = ['accommodates', 'bedrooms', 'bathrooms',
            'minimum_nights', 'latitude', 'longitude',
            'availability_365', 'number_of_reviews', 'reviews_per_month']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')

# Save splits
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)
print('Splits saved.')

Splits saved.
